<a href="https://colab.research.google.com/github/lokitheeditor697-create/Pathole/blob/main/train_pothole_yolov8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛣️ Smart City Road-Defect & Pothole Detection — YOLOv8 Training Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lokitheeditor697-create/Pathole/blob/main/train_pothole_yolov8.ipynb)

This notebook trains a custom **YOLOv8** model on a verified public pothole dataset for real-time onboard edge detection.

### Step 1: Check GPU & Install Dependencies

In [ ]:
!nvidia-smi
!pip install -q ultralytics huggingface_hub opencv-python matplotlib

### Step 2: Download Verified Pothole Dataset (Full Real Images via HF Hub)

In [ ]:
import os
import shutil
from huggingface_hub import snapshot_download

# 1. Clean previous attempts
if os.path.exists('/content/pothole_dataset'):
    shutil.rmtree('/content/pothole_dataset')

# 2. Download genuine binary images and labels (bypasses Git LFS pointer text files)
print('📥 Downloading verified pothole dataset...')
snapshot_download(
    repo_id='Ryukijano/Pothole-detection-Yolov8',
    repo_type='dataset',
    local_dir='/content/pothole_dataset',
    max_workers=8
)

# 3. Configure dataset.yaml with absolute Colab paths
yaml_path = '/content/pothole_dataset/data.yaml'
with open(yaml_path, 'w') as f:
    f.write('''path: /content/pothole_dataset
train: train/images
val: valid/images
test: test/images

names:
  0: pothole
''')

print('✅ Real dataset downloaded and configured successfully!')
print(open(yaml_path).read())

### Step 3: Train YOLOv8 Model on GPU

In [ ]:
from ultralytics import YOLO

# Initialize base YOLOv8 nano model
model = YOLO('yolov8n.pt')

# Train for 50 epochs on Tesla T4 GPU
results = model.train(
    data='/content/pothole_dataset/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    patience=10,
    name='pothole_yolov8_model'
)

### Step 4: Evaluate Model Performance (mAP, Loss & Confusion Matrix)

In [ ]:
import matplotlib.pyplot as plt
import cv2

# Validate the fine-tuned model
metrics = model.val()
print(f'Validation mAP50: {metrics.box.map50:.4f}')
print(f'Validation mAP50-95: {metrics.box.map:.4f}')

# Display training loss curves & confusion matrix
results_img = cv2.imread('runs/detect/pothole_yolov8_model/results.png')
if results_img is not None:
    plt.figure(figsize=(16, 10))
    plt.imshow(cv2.cvtColor(results_img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title('Training Loss & Validation Metrics')
    plt.show()

### Step 5: Test Model on Test Images

In [ ]:
# Run prediction on test set
preds = model.predict(source='/content/pothole_dataset/test/images', conf=0.35, save=True)
print('Predictions saved to runs/detect/predict/')

### Step 6: Download the Trained Model (`best.pt`)

In [ ]:
from google.colab import files
files.download('runs/detect/pothole_yolov8_model/weights/best.pt')